In [ ]:
# Definicion de nombres de catalogo y esquema
CATALOGO = "workspace"
ESQUEMA = "ingenieria_datos_tunombre"

TABLA_BRONZE = f"{CATALOGO}.{ESQUEMA}.ventas_bronze"
TABLA_SILVER = f"{CATALOGO}.{ESQUEMA}.ventas_silver"
TABLA_RESUMEN = f"{CATALOGO}.{ESQUEMA}.ventas_por_categoria"


In [ ]:
from pyspark.sql import functions as F

# Lectura de la tabla bronze
df_bronze = spark.read.table(TABLA_BRONZE)

# Registro del conteo inicial
conteo_inicial = df_bronze.count()
print(f"Cantidad de registros antes de la limpieza: {conteo_inicial}")


In [ ]:
# Eliminacion de registros identicos duplicados
df_sin_duplicados = df_bronze.dropDuplicates()

# Aplicacion de reglas de calidad:
# 1. Producto no nulo ni vacio
# 2. Cantidad mayor a cero
# 3. Precio unitario mayor a cero
df_filtrado = df_sin_duplicados.filter(
    F.col("producto").isNotNull() &
    (F.trim(F.col("producto")) != "") &
    (F.col("cantidad") > 0) &
    (F.col("precio_unitario") > 0)
)

# Calculo de la columna derivada total_venta
df_silver = df_filtrado.withColumn(
    "total_venta",
    F.round(F.col("cantidad") * F.col("precio_unitario"), 2)
)

# Registro del conteo final limpio
conteo_final = df_silver.count()
print(f"Cantidad de registros despues de la limpieza: {conteo_final}")


In [ ]:
# Visualizacion del DataFrame limpio
display(df_silver)


In [ ]:
# Persistencia idempotente de la capa silver
# El modo overwrite evita la duplicacion de filas en reejecuciones
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(TABLA_SILVER)


In [ ]:
# Agregacion del total vendido por categoria
df_resumen = df_silver.groupBy("categoria").agg(
    F.round(F.sum("total_venta"), 2).alias("total_vendido")
).orderBy(F.col("total_vendido").desc())

# Visualizacion del resumen
display(df_resumen)


In [ ]:
# Persistencia idempotente de la tabla de resumen por categoria
df_resumen.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(TABLA_RESUMEN)
